<font color=red>**Danger zone:**</font> you'll be fine-tuning a model to generate positive, negative or even toxic reviews. We'll be doing this for fun, but this is also the technique for [review bombing](https://en.wikipedia.org/wiki/Review_bomb), bot farms on social media and other less than dignified stuff. It is ultimately your decision how you apply this knowledge, but before you choose, ask yourself: is this why you chose to learn ML?


# LLMs Alignment with Reinforcement Learning from human feedback (RLHF).

_based on the [original notebook](https://github.com/antndlcrx/oxford-llms-workshop/blob/main/materials/seminars/day_3/8_LLMs%20alignment%20with%20RLHF.ipynb) by Ilya Boytsov for the Oxford LLMs workshop_



In this session, you're gonna fine-tune a language model with reinforcement learning to make it generate good (or bad) reviews.

To perform RL-based fine-tuning, we'll use a new (in this course) library called [Transformer Reinforcement Learning (TRL)](https://huggingface.co/docs/trl). TRL implements the main reinforcement learning components of RLHF: reward modeling and fine-tuning with PPO.

![img](https://huggingface.co/datasets/trl-internal-testing/example-images/resolve/main/images/TRL-readme.png)

### Tutorial: align the model to generate positive movie reviews

To see how TRL works, we'll use it to align GPT2 on IMDB dataset to generate positive (or negative) movie reviews. In fact, __it's your choice whether you want positive or negative reviews.__

But before you choose, let's take a look at the baseline model: a GPT-2 fine-tuned on generating arbitrary movie reviews.

In [78]:
import torch
import transformers
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
main_tokenizer = transformers.AutoTokenizer.from_pretrained("lvwerra/gpt2-imdb")
main_model = transformers.AutoModelForCausalLM.from_pretrained("lvwerra/gpt2-imdb", device_map=device)

In [2]:
inputs = main_tokenizer("The movie", return_tensors='pt').to(device)
generated_ids = main_model.generate(**inputs, max_new_tokens=50, do_sample=True)
print("\nGenerated text:", main_tokenizer.decode(generated_ids.flatten().cpu().numpy().tolist()))

Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
C:\Users\aleksandr.egorov\ml\.venv\lib\site-packages\transformers\models\gpt2\modeling_gpt2.py:545: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at ..\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:455.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(



Generated text: The movie was made in the early 1990s, the first years of the Americanization of film. While most of this movie is quite funny and makes you laugh, the bad guys are the main reasons why this movie is so bad. I have a confession to


If you run this cell a couple of times, you'll see that the model generates both positive, negative and neutral reviews in some proportion. What we're gonna do next is teach the model to generate more positive (or negative) reviews.

Similarly to InstructGPT, we're gonna do that in 2 stages:
- **train a reward model** to assign higher values to positive (or negative) reviews
- fine-tune the language model to **maximize that reward using [proximal policy optimization](https://openai.com/research/openai-baselines-ppo)**



## Stage 1: train a reward model

First, we'll train a BERT-like model as our reward model. We'll generate a synthetic pairwise rankings to emulate human rankings.

__Q:__ why do I need a reward model? Can I just use a pre-trained sentiment classifier? <br> __A:__ Yes, you can - but that only works for movie reviews. But this tutorial will teach you how to do RLHF for any kind objective.


__If you actually want to maximize sentiment (or other "label") instead of human preferences, train reward model as a classifier! (see week5)__


In [3]:
# We'll be fine-tuning a small BERT-like model for now. Please try other models for the main assignment.
reward_model = transformers.AutoModelForSequenceClassification.from_pretrained("distilbert-base-cased", device_map=device)
reward_tokenizer = transformers.AutoTokenizer.from_pretrained("distilbert-base-cased")

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


__Note that__ the reward model has a separate tokenizer, different from the main model. They don't need to be the same for RLHF fine-tuning.

In [75]:
# To train a reward model, you need a dataset (or generator) of positive-negative pairs.
# Each training sample should be a dict with 4 keys:
#  - input_ids_chosen, attention_mask_chosen = tokenizer("A sentence that human labeler likes more")
#  - input_ids_rejected, attention_mask_rejected = tokenizer("A sentence that human labeler likes less")

import torch
import datasets

class IMDBPairwiseDataset(torch.utils.data.Dataset):
    """ A dataset of all possible pairs of chosen and texts in TRT reward training format """
    def __init__(self, imdb, tokenizer, accepted_label: int):
        super().__init__()
        self.tokenizer = tokenizer
        self.chosen_texts = [row['text'] for row in imdb if row['label'] == accepted_label]
        self.rejected_texts = [row['text'] for row in imdb if row['label'] != accepted_label]
        self.column_names = ["input_ids_chosen", "attention_mask_chosen", "input_ids_rejected", "attention_mask_rejected"]
        assert self.chosen_texts, f"no texts with label {accepted_label}"
        print(f"Found {len(self.chosen_texts)} chosen and {len(self.rejected_texts)} rejected texts, {len(self)} pairs")

    def __len__(self):
        return len(self.chosen_texts) * len(self.rejected_texts)  # all pairs

    def __getitem__(self, index: int):
        chosen = self.tokenizer(self.chosen_texts[index // len(self.chosen_texts)], truncation=True)
        rejected = self.tokenizer(self.rejected_texts[index % len(self.chosen_texts)], truncation=True)
        return dict(input_ids_chosen=chosen['input_ids'], attention_mask_chosen=chosen['attention_mask'],
                    input_ids_rejected=rejected['input_ids'], attention_mask_rejected=rejected['attention_mask'])

In [76]:
TARGET_LABEL = 1   # and make sure it works by reviewing the sample printed below
imdb = datasets.load_dataset("imdb", split='train')
reward_data = IMDBPairwiseDataset(imdb, reward_tokenizer, accepted_label=TARGET_LABEL)

sample = reward_data[31337]
print('CHOSEN:', reward_tokenizer.decode(sample['input_ids_chosen']))
print('REJECTED:', reward_tokenizer.decode(sample['input_ids_rejected']))

Found 12500 chosen and 12500 rejected texts, 156250000 pairs
CHOSEN: [CLS] Lars Von Trier is never backward in trying out new techniques. Some of them are very original while others are best forgotten. < br / > < br / > He depicts postwar Germany as a nightmarish train journey. With so many cities lying in ruins, Leo Kessler a young American of German descent feels obliged to help in their restoration. It is not a simple task as he quickly finds out. < br / > < br / > His uncle finds him a job as a night conductor on the Zentropa Railway Line. His job is to attend to the needs of the passengers. When the shoes are polished a chalk mark is made on the soles. A terrible argument ensues when a passenger ' s shoes are not chalked despite the fact they have been polished. There are many allusions to the German fanaticism of adherence to such stupid details. < br / > < br / > The railway journey is like an allegory representing man ' s procession through life with all its trials and tribulat

We'll be using `trl.RewardTrainer` - a special case of `transformers.Trainer` that you used in the past. `RewardTrainer` accepts the same format of training arguments (e.g. batch size, gradient checkpointing) as before, except that it trains the model for the pairwise reward objective from [the InstructGPT paper](https://arxiv.org/pdf/2203.02155.pdf):

![img](https://i.imgur.com/2JzNAPs.png)

Note that the model itself does not score pairs: it processes chosen ($y_w$) and rejected ($y_l$) samples independently. To minimize this loss, the reward model needs to score chosen sample higher than the rejected one. Note that the formula also assumes some context $x$, which is useful for seq2seq tasks. In our case of movie reviews, $x$ is empty.

In [6]:
import trl

training_args = trl.RewardConfig(  # like transformers.TrainingArguments
    output_dir="reward_model",
    per_device_train_batch_size=32,
    gradient_accumulation_steps=1,
    learning_rate=1.41e-5,
    max_steps=1_000,              # note: training may need more than 1k steps
    logging_steps=50,
    gradient_checkpointing=True,  # reduce memory usage but train ~30% slower
    gradient_checkpointing_kwargs={"use_reentrant": False},
    fp16=True                     # disable this on CPU or on very old GPUs
    # you may add any other hyperparameters that you found useful in weeks 5-7
)

trainer = trl.RewardTrainer(
    model=reward_model,
    args=training_args,
    tokenizer=reward_tokenizer,
    train_dataset=reward_data,
    peft_config=None,  # optionally, you may tune with LoRA, prompt-tuning, etc
)

trainer.train()

C:\Users\aleksandr.egorov\ml\.venv\lib\site-packages\trl\trainer\reward_trainer.py:182: UserWarning: When using RewardDataCollatorWithPadding, you should set `max_length` in RewardConfig. It will be set to `512` by default, but you should do it yourself in the future.
  warnings.warn(
C:\Users\aleksandr.egorov\ml\.venv\lib\site-packages\trl\trainer\reward_trainer.py:199: UserWarning: When using RewardDataCollatorWithPadding, you should set `remove_unused_columns=False` in your RewardConfig we have set it for you, but you should do it yourself in the future.
  warnings.warn(
max_steps is given, it will override any value given in num_train_epochs
wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.
wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.
wandb: Currently l

You're using a DistilBertTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
C:\Users\aleksandr.egorov\ml\.venv\lib\site-packages\transformers\tokenization_utils_base.py:2847: UserWarning: `max_length` is ignored when `padding`=`True` and there is no truncation strategy. To pad to max length, use `padding='max_length'`.
  warnings.warn(
Could not estimate the number of tokens of the input, floating-point operations will not be computed


Step,Training Loss
50,0.509300
100,0.181500
150,0.152200
200,0.122900
250,0.123400
300,0.087300
350,0.077400
400,0.072100
450,0.099100
500,0.068700


C:\Users\aleksandr.egorov\ml\.venv\lib\site-packages\transformers\tokenization_utils_base.py:2847: UserWarning: `max_length` is ignored when `padding`=`True` and there is no truncation strategy. To pad to max length, use `padding='max_length'`.
  warnings.warn(


TrainOutput(global_step=1000, training_loss=0.10799304509162903, metrics={'train_runtime': 1339.0845, 'train_samples_per_second': 23.897, 'train_steps_per_second': 0.747, 'total_flos': 0.0, 'train_loss': 0.10799304509162903, 'epoch': 0.00020479997902848215})

In [7]:
reward_model.gradient_checkpointing_disable()
reward_model.eval()

DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(28996, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): MultiHeadSelfAttention(
            (dropout): Dropout(p=0.1, inplace=False)
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)
 

In [ ]:
with open("./reward_model_checkpoint.pth", "wb") as fp:
    torch.save(reward_model.state_dict(), fp)

### Sanity-check the reward model (1 point)

Let's check how our reward model performs.

__Your task__ is to measure how often does your reward model can rank a pair of (chosen and rejected) reviews correctly. Please measure this separately for train data (`imdb`) and a separate test set loaded below.

In [8]:

for sample_index in 45, 16000:
  print('TEXT:', imdb[sample_index]['text'])
  inputs = reward_tokenizer(
      imdb[sample_index]['text'], truncation=True, return_tensors='pt').to(device)
  with torch.no_grad():
    reward = reward_model(**inputs).logits[0, 0].item()
    print("REWARD:", reward)
  print('LABEL:', imdb[sample_index]['label'])
  print()

# note: your reward model may produce different absolute rewards.
# This is fine as long as the rewards are ordered correctly (most of the time)

TEXT: This movie sucked. It really was a waste of my life. The acting was atrocious, the plot completely implausible. Long, long story short, these people get "terrorized" by this pathetic "crazed killer", but completely fail to fight back in any manner. And this is after they take a raft on a camping trip, with no gear, and show up at a campsite that is already assembled and completely stocked with food and clothes and the daughters headphones. Additionally, after their boat goes missing, they panic that they're stuck in the woods, but then the daughters boyfriend just shows up and they apparently never consider that they could just hike out of the woods like he did to get to them. Like I said, this movie sucks. A complete joke. Don't let your girlfriend talk you into watching it.
REWARD: -5.28125
LABEL: 0

TEXT: Good: Engaging cinematic firefights, great presentation, vehicles are actually fun to drive, fairly appealing multiplayer, faithful to the movie, and the list goes on.<br /><

In [17]:
import pandas as pd
def reward_sanity_check(imdb, labels=None):
    if labels is None:
        labels = []
        rewards = []
    for i in imdb:
        labels.append(i["label"])
        inputs = reward_tokenizer(
          i['text'], truncation=True, return_tensors='pt').to(device)
        with torch.no_grad():
            reward = reward_model(**inputs).logits[0, 0].item()
            rewards.append(reward)
    return labels, rewards 
    
labels, rewards = reward_sanity_check(imdb)    



In [29]:
stat = pd.DataFrame([labels, rewards]).T.rename(columns={0:"label", 1:"reward"})
stat

,label,reward
0,0.0,-1.159180
1,0.0,-5.218750
2,0.0,-5.058594
3,0.0,-2.062500
4,0.0,-5.039062
...,...,...
24995,1.0,-1.995117
24996,1.0,2.736328
24997,1.0,4.871094
24998,1.0,1.620117


### Train Так как целевая метка 1, то 0 в основном должна иметь отрицательный reward:

In [37]:
stat[(stat["label"] == 0) & (stat["reward"]<=0)].shape[0]/stat[(stat["label"] == 0)].shape[0]

0.95072

### Train Так как целевая метка 1, то 1 в основном должна иметь положительный reward:

In [39]:
stat[(stat["label"] == 1) & (stat["reward"]>0)].shape[0]/stat[(stat["label"] == 1)].shape[0]

0.91904

In [42]:
imdb_test = datasets.load_dataset("imdb", split='test')
labels, rewards = reward_sanity_check(imdb_test)
# <a whole lot of your code here, feel free to spit it as you see fit>

### Test Так как целевая метка 1, то 0 в основном должна иметь отрицательный reward:

In [43]:
stat = pd.DataFrame([labels, rewards]).T.rename(columns={0:"label", 1:"reward"})
stat[(stat["label"] == 0) & (stat["reward"]<=0)].shape[0]/stat[(stat["label"] == 0)].shape[0]

0.92616

### Test Так как целевая метка 1, то 1 в основном должна иметь положительный reward:

In [45]:
stat[(stat["label"] == 1) & (stat["reward"]>0)].shape[0]/stat[(stat["label"] == 1)].shape[0]

0.89648

### Высокая доля правильных ответов

### Reward-guided generation (1 point)

If you did everything right, by now you should have a decent reward model. Before we use it for reinforcement learning, let's see if we can align model samples without any training.

To do so, you can use reward-guided inference: __generate N=16 samples, then select the one with the highest reward__ (according to your reward model).

For this problem, it's on you to demonstrate whether or not your code works. Find at least 5 neutral prompts such as "This movie is" (...), generate samples, rank them based on reward and show which samples get the highest reward.

Note: it is faster to generate samples in parallel, rather than sequentially, as follows:




In [57]:
inputs = main_tokenizer(["It was"] * 5, return_tensors='pt').to(device)
for candidate in main_model.generate(**inputs, max_new_tokens=50, do_sample=True):
  print("Sample:", main_tokenizer.decode(candidate.flatten().cpu().numpy().tolist()))

Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


Sample: It was like this movie is trying to show you something about the Vietnam war. You see what it was like, and you know it was a terrible one-sided movie. This movie is about an American who went into war with no real plan and he didn
Sample: It was such a shock to me, when I heard all the other people talk about it.<br /><br />However, this was one of my first movies to use a real-time projection system because I felt they'd use this technology much better in
Sample: It was an amazing experience. I highly recommend this movie for all the viewers that want some laughs, with it's realistic and interesting characters. Also, this movie is worth catching on cable TV/demand TV. It definitely has some real laughs. However, in
Sample: It was a fun film, if hardly the most original of the "R-rated series," "Hollywood Movie" with many of the early-twenties teen-comic-fan boys and, to an extent, kids with more mature sensibilities
Sample: It was wonderful film. I don't really pay attention 

In [15]:
def generate_examples(prompt:str, N:int):
    examples = []
    inputs = main_tokenizer([prompt] * N, return_tensors='pt').to(device)

    return [main_tokenizer.decode(candidate.flatten().cpu().numpy().tolist()) for candidate in main_model.generate(**inputs, max_new_tokens=50, do_sample=True)]

def get_max_reward(examples):
    rewards = {}
    for i in examples:
        inputs = reward_tokenizer(
              i, truncation=True, return_tensors='pt').to(device)
        with torch.no_grad():
            reward = reward_model(**inputs).logits[0, 0].item()
            rewards[i] = reward
    max_reward_text = max(rewards, key=rewards.get)
    return {max_reward_text: rewards[max_reward_text]}


main_model.generation_config = GenerationConfig()
examples = generate_examples("It was", 16)

In [87]:
examples = generate_examples("It was", 16)
max_reward = get_max_reward(examples)
max_reward

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


{"It was very easy to catch up with the other episodes. They were all very good, both with the original show and now with Season 9. I loved the show, which was such a different show and the story was great. I've seen that episode twice": 4.625}

In [89]:
examples = generate_examples("The movie", 16)
max_reward = get_max_reward(examples)
max_reward

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


{"The movie is so well-made it's easy to understand why so many critics have picked it up instead of reworking it for TV. It's a wonderfully constructed, well paced action drama with very good dialogue, action, suspense, and an open ending that": 4.80078125}

In [90]:
examples = generate_examples("The beginning of the film", 16)
max_reward = get_max_reward(examples)
max_reward

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


{'The beginning of the film is very touching and touching. The story line in the film, which involves that of the three young parents (from various other backgrounds), is pretty easy to relate to and, as you move to the next part and the finale, it gets even better': 4.61328125}

In [93]:
examples = generate_examples("By the end of the movie", 16)
max_reward = get_max_reward(examples)
max_reward

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


{'By the end of the movie, we get to see how the kids got to be such good and so powerful. This movie is a testament to the power of movies. This could have been a good movie, if the story had been like this. The acting was perfect and the': 4.6953125}

In [95]:
examples = generate_examples("I want to tell you", 16)
max_reward = get_max_reward(examples)
max_reward

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


{'I want to tell you that I loved this film. The scene with the girls (who were very talented in front of a camera and made perfect faces) making their voices are amazing. The script is well written with lots of action and comedy moments along with some beautiful images of': 4.78125}

### Отзывы положительные - как и выбрано в начале.

# Stage 2: fine-tune the main model with RL


For this tutorial, we will optimize GPT2 to produce positive IMDB movie reviews using the reward model you trained above.

Unlike supervised fine-tuning, RL allows model to generate it's own sentences on each training step. Then, it calculates the reward of those specific sentences, and finally, updates the model to increase the probability of sentences with high reward.

Thus, each RLHF consists of three stages: __Rollout__, __Evaluation__ and __Update__

<div style="text-align: center">
<img src='https://huggingface.co/datasets/trl-internal-testing/example-images/resolve/main/images/gpt2_bert_training.png' width='600'>

The update stage depends on the specific RL algorithm. We'll be using Proximal Policy Optimization, or [PPO](https://arxiv.org/abs/1707.06347), similarly to what was used for InstructGPT.

Before we run those 3 stages, however, we need to create a dataset of "queries" - partial reviews in our case.

In [79]:
# Note: this code is specific to IMDB; you will need to re-write it for other tasks
import trl
imdb_for_rlhf = imdb.filter(lambda row: len(row['text']) > 200, batched=False)
imdb_for_rlhf = imdb_for_rlhf.remove_columns(['label'])
sample_length = trl.core.LengthSampler(2, 8)  # use the first 2-8 tokens as query

def select_query_and_tokenize(sample):
    query_ids = main_tokenizer.encode(sample["text"])[: sample_length()]
    sample["query"] = main_tokenizer.decode(query_ids)  # query is the only required column
    sample["input_ids"] = query_ids  # to avoid re-tokenizing later
    return sample  # we do not need the rest - it will be generated by the model

imdb_for_rlhf = imdb_for_rlhf.map(select_query_and_tokenize, batched=False)
imdb_for_rlhf.set_format(type="torch")

Map:   0%|          | 0/24895 [00:00<?, ? examples/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (1168 > 1024). Running this sequence through the model will result in indexing errors


Next, let's prepare your reward model to predict rewards on whatever reviews were generated. Note that we use plaintext reviews because main model uses a different tokenizer from the reward model.

In [7]:
with open("./reward_model_checkpoint.pth", "rb") as fp:
    state_dict = torch.load(fp, map_location="cpu")
    reward_model.load_state_dict(state_dict)
    reward_model = reward_model.to(device)

In [22]:
from typing import List
def compute_reward(texts: List[str]) -> torch.Tensor:
  inputs = reward_tokenizer(texts, truncation=True, padding=True, return_tensors='pt').to(device)
  with torch.no_grad():
    return reward_model(**inputs).logits[:, 0]

In [9]:
compute_reward([imdb[45]['text'], imdb[16000]['text']])  # test on human-written reviews

tensor([-5.3548,  4.3998], device='cuda:0')

Finally, we move to RL training. In this tutorial, we'll train LoRA adapters and not the full model.

In [10]:
import peft
peft_config = peft.LoraConfig(
    task_type=peft.TaskType.CAUSAL_LM, r=32, lora_alpha=32, lora_dropout=0.0, inference_mode=False
)

# reload main model as AutoModelForCausalLMWithValueHead - with an extra head needed for PPO
main_tokenizer = transformers.AutoTokenizer.from_pretrained("lvwerra/gpt2-imdb")
main_tokenizer.pad_token = main_tokenizer.eos_token

main_model = trl.AutoModelForCausalLMWithValueHead.from_pretrained("lvwerra/gpt2-imdb", device_map=device)
main_model = peft.get_peft_model(main_model, peft_config, adapter_name='default')
main_model.print_trainable_parameters()


C:\Users\aleksandr.egorov\ml\.venv\lib\site-packages\huggingface_hub\file_download.py:797: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
C:\Users\aleksandr.egorov\ml\.venv\lib\site-packages\peft\tuners\lora.py:475: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(


trainable params: 1,179,648 || all params: 125,620,225 || trainable%: 0.9390589771670923


Same as before, trl has a special type of trainer that minimize PPO-specific pseudo-loss. You can read more on this trainer [here](https://huggingface.co/docs/trl/main/en/ppo_trainer).

In [11]:
training_args = trl.PPOConfig(
    model_name=main_model.config._name_or_path,
    gradient_accumulation_steps=1,
    learning_rate=1.41e-5,
    ppo_epochs=4,
    mini_batch_size=64,
    batch_size=64,# PPO performs this many updates per training batch
)

ppo_trainer = trl.PPOTrainer(
    training_args, model=main_model.model, tokenizer=main_tokenizer, ref_model=None,
    dataset=imdb_for_rlhf, data_collator=lambda data: dict((key, [d[key] for d in data]) for key in data[0])
)  # note: we pass main_model.model because PPOTrainer checks for one of several supported model types ...
# ... main_model.model is a model with adapters, which is supported. main_model itself is a wrapper that is not supported

In [12]:
from tqdm.auto import tqdm
max_steps = 50   # can be insufficient for some tasks - watch your learning curves
generation_kwargs = dict(
    min_length=-1, max_new_tokens=120, do_sample=True, top_k=0, top_p=1.0, pad_token_id=main_tokenizer.eos_token_id,
eos_token_id=main_tokenizer.eos_token_id,)
#                                  ^-- task-specific parameter!
with tqdm(enumerate(ppo_trainer.dataloader), total=max_steps) as progressbar:
  # note: ppo_trainer.dataloader is just a regular dataloader of queries, no RL-specific magic :)
  for epoch, batch in progressbar:
    if epoch >= max_steps:
        break

    # Rollout stage: generate continuations from batch queries using main_model
    response_tensors = ppo_trainer.generate(batch['input_ids'], **generation_kwargs)
    # ^-- list of tensors of token ids from main model tokenizer

    # de-tokenize responses to strings (since reward model uses a different tokenizer)
    batch["response"] = [main_tokenizer.decode(response.squeeze()) for response in response_tensors]
    # note: response_tensors already contain query tokens, so we don't need to add queries manually.
    # This may not be true for other tasks: check this manually by viewing batch["response"] and batch["query"]
    # Evaluation stage
    rewards = compute_reward(batch['response'])
    # Update stage
    stats = ppo_trainer.step(batch['input_ids'], response_tensors, list(rewards.split(1)))
    stats['rewards/mean'] = rewards.mean().item()

    print("-" * 30, 'STEP', epoch, '-' * 30)
    print(f'rewards/mean:\t{stats["rewards/mean"]:.9f}\t<---- average reward over this batch (higher=better, noisy)')
    print(f'ppo/returns/mean:\t{stats["ppo/returns/mean"]:.9f}\t<---- model-estimated average discounted reward')
    print(f'objective/kl:\t{stats["objective/kl"]:.9f}\t<---- how far we are from the original model (regularizer)')
    print()

    ppo_trainer.log_stats(stats, batch, list(rewards.split(1)))

  0%|          | 0/50 [00:00<?, ?it/s]

You're using a GPT2TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


------------------------------ STEP 0 ------------------------------
rewards/mean:	-0.762633264	<---- average reward over this batch (higher=better, noisy)
ppo/returns/mean:	0.856959820	<---- model-estimated average discounted reward
objective/kl:	0.000000000	<---- how far we are from the original model (regularizer)

------------------------------ STEP 1 ------------------------------
rewards/mean:	-0.194964558	<---- average reward over this batch (higher=better, noisy)
ppo/returns/mean:	1.109931469	<---- model-estimated average discounted reward
objective/kl:	0.004979739	<---- how far we are from the original model (regularizer)

------------------------------ STEP 2 ------------------------------
rewards/mean:	-0.071071565	<---- average reward over this batch (higher=better, noisy)
ppo/returns/mean:	1.122048020	<---- model-estimated average discounted reward
objective/kl:	0.009733500	<---- how far we are from the original model (regularizer)

------------------------------ STEP 3 --

In [43]:
ref_model = trl.AutoModelForCausalLMWithValueHead.from_pretrained("lvwerra/gpt2-imdb", device_map=device)
new = 0
old = 0
for i in range(10):

    inputs = main_tokenizer(["This film"] * 16, return_tensors='pt').to(device)

    new += compute_reward([main_tokenizer.decode(candidate.flatten().cpu().numpy().tolist()) for candidate in main_model.model.generate(**inputs, max_new_tokens=50, do_sample=True)]).mean()
    old += compute_reward([main_tokenizer.decode(candidate.flatten().cpu().numpy().tolist()) for candidate in ref_model.generate(**inputs, max_new_tokens=50, do_sample=True)]).mean()

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end gene

In [46]:
print("mean reward before:", old/10)
print("mean reward after:", new/10)


mean reward before: tensor(0.6024, device='cuda:0')
mean reward after: tensor(1.4407, device='cuda:0')


## Main assignment - <u>actually</u> train the model (8 points)


Your main task for this week is to use the RLHF pipeline to train a model for a reward of your choice. Here's what you can choose from:

__A. Toxicity fine-tuning:__ train the model to be less (or more!) toxic. For this task, you may use the data from [jigsaw toxic comments](https://www.kaggle.com/c/jigsaw-toxic-comment-classification-challenge) and [lmsys/toxic-chat](https://huggingface.co/datasets/lmsys/toxic-chat),  or any other source. Alternatively, you may use toxicity scores from [oasst1](https://huggingface.co/datasets/OpenAssistant/oasst1).


__B. Actual human feedback:__ use one of the existing datasets with pairwise human feedback to align your langauge model. You may use [anthropic's hh-rlhf](https://huggingface.co/datasets/Anthropic/hh-rlhf), [OpenAssistant dataset](https://huggingface.co/datasets/OpenAssistant/oasst1) or any other data you see fit. You may also turn the tables and train the model to [minimize](https://habrastorage.org/getpro/geektimes/post_images/ac7/2ad/827/ac72ad82767d4132164a4b6b76196c42.jpg) human preferences, as long as your model does not degrade to gibberish.

__C. Controlled generation:__ Instead of training a reward model from human feedback, you may define the reward function as the text length (longer or shorter) or number of times the model uses specific words (e.g. "sorry", "apologize"). If you choose specific words, make sure the model generates them at least sometimes.

__Alternatively,__ you may choose a different task. However, unless your task is very similar to one of the above, there is a chance that it will be **significantly** harder to solve, requiring orders of magnitude more compute and tuning. If you are in doubt, please ask the course staff. If they are AFK (again >.<), please prefer one of the recommended tasks.


#### General tips & tricks


Things to look out for:
- during PPO stage, the reward model should be in eval mode (dropout disabled)
- make sure max_length and max_new_tokens are enough for your chosen dataset - at least most of the time
- when in doubt, view the data manually or inspect how the model performs on a few samples


We highly recommend that you manually check the performance after each sub-stage:
1. when you assembled the pairwise dataset, inspect a couple of from of *your* dataset class and detokenize them. Make sure that you-the-human understand why one sample was accepted and the other - rejected. At least most of the time. This also lets you spot tokenization/truncation errors.
2. after you trained a reward model, measure how accurate this model is in isolation. If your reward model is poor, any subsequent RLHF will also fail.
3. once you've trained the main model with RL, ask it to generate examples and explore how well it does. If it produces an obviously bad output, check if the reward model assigns high reward to that output. If yes, reward model is the culprit; if no, it's a question of better/longer PPO training.

__It is also a good idea to periodically print samples during training.__

__When stuck, simplify the problem.__ If you've spent a several hours enchanting the reward model but it still won't budge, try switching to a simple subtask. For instance, if you're training on hh-rlhf, try limiting it the dataset to 10% of the shortest sequences - they are typically easier to learn.


## Assignment stages (and grading)

Regardless of the specific task you chose, your solution needs to contain several parts that will be graded separately.


#### Stage 1: reward model (4 points)

Construct a dataset for training the reward model on your problem. Then, train a reward model on that dataset and evaluate how well can your model predict preferences on a hold-out (test) subset of your data.

Please make sure that the part of your notebook where you evaluate reward model is clearly visible and reasonably easy to read. And for all that is holy, do not call it IMDB unless it actually **is** data of imdb movie reviews :)

__Not all tasks require a reward model for later PPO fine-tuning.__ For instance, there's no reason to train a reward model if your reward equals sentence length. Likewise, toxicity reward can be estimated with a pre-trained toxicity classifier. __If your task does not require training a reward model, please train an unrelated model on [hh-rlhf](https://huggingface.co/datasets/Anthropic/hh-rlhf) as though you were solving assignment version B.__ This is for grading purposes only, you won't use this model for stage 2.


#### Stage 2: RL fine-tuning (4 points)

Once the reward model is ready - or you can compute rewards without a model - it is time to maximize that reward with PPO. Optionally, you may replace PPO with another RL algorithm (or unlikelihood learning scheme), but only if you're feeling adventurous.


First, you need to choose a language model to be fine-tuned. You may choose any model, but make sure that your model **can** generate the data in your format. For instance, [Mistral-7B](https://huggingface.co/mistralai/Mistral-7B-v0.1) is a general purpose LM and may (or may not) need prompt engineering to generate chat assistant responses. For that reason, it is best if you **do not use `"lvwerra/gpt2-imdb"` unless you're generating only movie reviews**.



There are two "difficulty modes" for this task:
For the **easy mode**, use [gpt2-large](https://huggingface.co/gpt2-large) or [opt-1.3b](https://huggingface.co/facebook/opt-1.3b) with minimal code changes.
If you want the **Hard mode:** use a larger (e.g. 7B) model in combination with `load_in_4bit` and LoRA, the same way we did last week.
Some reasonable model choices are [LLaMA-7B](https://huggingface.co/Enoch/llama-7b-hf), [Falcon-7b](https://huggingface.co/tiiuae/falcon-7b), [Mistral-7B](https://huggingface.co/mistralai/Mistral-7B-v0.1) for general-purpose LM or [guanaco-7b](https://huggingface.co/timdettmers/guanaco-7b), [vicuna-7b](https://huggingface.co/lmsys/vicuna-7b-v1.5) for chat-based tasks, though there are many more (see [leaderboard](https://huggingface.co/spaces/HuggingFaceH4/open_llm_leaderboard)). In the hard mode, you will need to modify the training arguments to enable 4-bit fine-tuning. Furthermore, your experiments will take somewhat longer to complete. On the plus side, your model will produce significantly better results.

__High reward is not enough!__ RL algorithms are famous for [cheating their reward functions](https://openai.com/research/faulty-reward-functions). To ensure that your model is actually doing what you want it to do, you will need some additional evaluation. To get the full grade, provide at least 20 side-by-side examples of your fine-tuned model vs original model predictions and a short summary.

Alternatively, you may provide 5 examples and some extrinsic evaluation metric over many examples. For instance, you may use a different pre-trained toxicity score for option A. When dealing with human preferences, you may choose to [enlist actual humans](https://toloka.ai/) or [ask GPT4/Claude](https://arxiv.org/pdf/2304.03277.pdf) to compare your model's predictions. For task C, when optimizing for simple rewards like sentence lengths, it is enough to compare histograms of rewards (e.g. average lengths).












# Reward model

### Я использую датасет: "Yelp/yelp_review_full" с отзывами. Отзывы проранжированы от 0 до 4, где 0 - плохо, 4 - отлично. Я убрал оценку 2(как среднюю) и объединил 0 и 1 как плохой отзыв, 3 и 4 как хороший. Отзывов очень много, для ускорения я взял небольшой процент для обучения и стратифицировал выборку, чтобы плохих и хороших отзывов было равное количество.

In [ ]:
#!pip install trl==0.11.3 transformers==4.45.2 

In [60]:
pip list

Package                    VersionNote: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 24.0 -> 24.3.1

-------------------------- ------------------

[notice] To update, run: python.exe -m pip install --upgrade pip

accelerate                 1.1.1


adjustText                 1.3.0
aiohappyeyeballs           2.4.3
aiohttp                    3.11.2
aiosignal                  1.3.1
albucore                   0.0.20
albumentations             1.4.21
annotated-types            0.7.0
antlr4-python3-runtime     4.9.3
anyio                      4.6.2.post1
argon2-cffi                23.1.0
argon2-cffi-bindings       21.2.0
arrow                      1.3.0
asttokens                  2.4.1
async-lru                  2.0.4
async-timeout              5.0.1
attrs                      24.2.0
axial_positional_embedding 0.2.1
babel                      2.16.0
beautifulsoup4             4.12.3
bleach                     6.2.0
bleuscore                  0.1.3
blis                       0.7.11
bs4                        0.0.2
catalogue                  2.0.10
certifi                    2024.8.30
cffi                       1.17.1
charset-normalizer         3.4.0
click                      8.1.7
cloudpathlib               0.20.0
cloudpickle         

In [1]:
from datasets import load_dataset

ds = load_dataset("Yelp/yelp_review_full", split='train[:9%]').filter(lambda example: example["label"]!=2 )

ds = ds.train_test_split(test_size=0.5, stratify_by_column="label")

dataset = ds.sort(column_names="label")["train"].select(range(1835, 23515))

In [2]:
import collections
collections.Counter(dataset['label'])

Counter({1: 6028, 3: 5597, 4: 5243, 0: 4812})

In [3]:
# We'll be fine-tuning a small BERT-like model for now. Please try other models for the main assignment.
import torch
import transformers
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
reward_model = transformers.AutoModelForSequenceClassification.from_pretrained("distilbert-base-cased", device_map=device)
reward_tokenizer = transformers.AutoTokenizer.from_pretrained("distilbert-base-cased")

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [4]:
class YelpPairwiseDataset(torch.utils.data.Dataset):
    """ A dataset of all possible pairs of chosen and texts in TRT reward training format """
    def __init__(self, ds, tokenizer, accepted_label: int):
        super().__init__()
        self.tokenizer = tokenizer
        self.chosen_texts = [row['text'] for row in ds if row['label'] > accepted_label]
        self.rejected_texts = [row['text'] for row in ds if row['label'] <= accepted_label]
        self.column_names = ["input_ids_chosen", "attention_mask_chosen", "input_ids_rejected", "attention_mask_rejected"]
        assert self.chosen_texts, f"no texts with label {accepted_label}"
        print(f"Found {len(self.chosen_texts)} chosen and {len(self.rejected_texts)} rejected texts, {len(self)} pairs")

    def __len__(self):
        return len(self.chosen_texts) * len(self.rejected_texts)  # all pairs

    def __getitem__(self, index: int):
        chosen = self.tokenizer(self.chosen_texts[index // len(self.chosen_texts)], truncation=True)
        rejected = self.tokenizer(self.rejected_texts[index % len(self.chosen_texts)], truncation=True)
        return dict(input_ids_chosen=chosen['input_ids'], attention_mask_chosen=chosen['attention_mask'],
                    input_ids_rejected=rejected['input_ids'], attention_mask_rejected=rejected['attention_mask'])

In [7]:
TARGET_LABEL = 1   # and make sure it works by reviewing the sample printed below
reward_data = YelpPairwiseDataset(dataset, reward_tokenizer, accepted_label=TARGET_LABEL)

sample = reward_data[10000]
print('CHOSEN:', reward_tokenizer.decode(sample['input_ids_chosen']))
print('REJECTED:', reward_tokenizer.decode(sample['input_ids_rejected']))

Found 10840 chosen and 10840 rejected texts, 117505600 pairs
CHOSEN: [CLS] Nice airport with good dining choices. The US Airways Lounge is great too. [SEP]
REJECTED: [CLS] Went in for the living social deal last Friday. It was busy during happy hour. Seemed like the spot to be! Although, once my boyfriend and I were seated and advised the server that we had the living social deal, the service was extremely disappointing. I understand that with coupons at restaurants servers don ' t always get tipped on the price before discount. I used to be a server and restaurant manager and have seen that happen a lot. The server was very robotic and not friendly at all. She set down our drinks and food like we were in her way and troubling her. The appetizers were fair for pub food. The pretzels were pretty tasty actually! We really enjoyed the beer sampler that we each got as well! My favorites were the hefeweizen and vanilla porter. Everyone else that worked there seemed very pleasant and friendl

In [8]:
import trl

training_args = trl.RewardConfig(  # like transformers.TrainingArguments
    output_dir="reward_model",
    per_device_train_batch_size=32,
    gradient_accumulation_steps=1,
    learning_rate=1.41e-5,
    max_steps=1_000,              # note: training may need more than 1k steps
    logging_steps=50,
    gradient_checkpointing=True,  # reduce memory usage but train ~30% slower
    gradient_checkpointing_kwargs={"use_reentrant": False},
    fp16=True                     # disable this on CPU or on very old GPUs
    # you may add any other hyperparameters that you found useful in weeks 5-7
)

trainer = trl.RewardTrainer(
    model=reward_model,
    args=training_args,
    tokenizer=reward_tokenizer,
    train_dataset=reward_data,
    peft_config=None,  # optionally, you may tune with LoRA, prompt-tuning, etc
)

trainer.train()

C:\Users\aleksandr.egorov\ml\.venv\lib\site-packages\trl\trainer\reward_trainer.py:182: UserWarning: When using RewardDataCollatorWithPadding, you should set `max_length` in RewardConfig. It will be set to `512` by default, but you should do it yourself in the future.
  warnings.warn(
C:\Users\aleksandr.egorov\ml\.venv\lib\site-packages\trl\trainer\reward_trainer.py:199: UserWarning: When using RewardDataCollatorWithPadding, you should set `remove_unused_columns=False` in your RewardConfig we have set it for you, but you should do it yourself in the future.
  warnings.warn(
max_steps is given, it will override any value given in num_train_epochs
wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.
Failed to detect the name of this notebook, you can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable c

You're using a DistilBertTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
C:\Users\aleksandr.egorov\ml\.venv\lib\site-packages\transformers\tokenization_utils_base.py:2847: UserWarning: `max_length` is ignored when `padding`=`True` and there is no truncation strategy. To pad to max length, use `padding='max_length'`.
  warnings.warn(
Could not estimate the number of tokens of the input, floating-point operations will not be computed


Step,Training Loss
50,0.432700
100,0.130400
150,0.080200
200,0.067400
250,0.064800
300,0.062000
350,0.052600
400,0.036800
450,0.045500
500,0.032200


C:\Users\aleksandr.egorov\ml\.venv\lib\site-packages\transformers\tokenization_utils_base.py:2847: UserWarning: `max_length` is ignored when `padding`=`True` and there is no truncation strategy. To pad to max length, use `padding='max_length'`.
  warnings.warn(


TrainOutput(global_step=1000, training_loss=0.06505316001176834, metrics={'train_runtime': 1216.1758, 'train_samples_per_second': 26.312, 'train_steps_per_second': 0.822, 'total_flos': 0.0, 'train_loss': 0.06505316001176834, 'epoch': 0.0002723274465216977})

In [9]:
reward_model.gradient_checkpointing_disable()
reward_model.eval()

DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(28996, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): MultiHeadSelfAttention(
            (dropout): Dropout(p=0.1, inplace=False)
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)
 

In [45]:
with open("./reward_review_model_checkpoint.pth", "wb") as fp:
    torch.save(reward_model.state_dict(), fp)

In [18]:
for sample_index in 45, 15000:
  print('TEXT:', dataset[sample_index]['text'])
  inputs = reward_tokenizer(
      dataset[sample_index]['text'], truncation=True, return_tensors='pt').to(device)
  with torch.no_grad():
    reward = reward_model(**inputs).logits[0, 0].item()
    print("REWARD:", reward)
  print('LABEL:', dataset[sample_index]['label'])
  print()

TEXT: We wanted to try some Japanese food. A nameless coworker of mine suggested Ichi Ban, located on Alma School and Baseline Rd. Apparently he told us that he had drove by it and thought it would be worth a try. He must have drove by it drunk. The appearance of the building is simply a monstrosity in and of itself. The architects must have been on acid when they designed it because it is such an eye sore. I keep hearing in my head, \"..don't judge a book by its cover\", yet my judgments usally end up being correct. \n\nWhen we walked into the establishment that is Ichi Ban there was nobody in there. I mean it was totally dead. At that moment I felt that if we continued to sit down and eat we might just die. Ichi Ban had Kitchen Nightmare written all over it. I mean, if when you walk into a restaurant and nobody is there take that as a sign from God not to eat. So we sat down and ordered some food. \n\nEverybody got soup. I guess it came with your meal. I didn't try the soup myself, b

In [19]:
import pandas as pd
def reward_sanity_check(ds, labels=None):
    if labels is None:
        labels = []
        rewards = []
    for idx, i in enumerate(ds):
        labels.append(i["label"])
        inputs = reward_tokenizer(
          i['text'], truncation=True, return_tensors='pt').to(device)
        with torch.no_grad():
            reward = reward_model(**inputs).logits[0, 0].item()
            rewards.append(reward)
    return labels, rewards 
    
label, rewards = reward_sanity_check(dataset)

In [20]:
stat = pd.DataFrame([label, rewards]).T.rename(columns={0:"label", 1:"reward"})
stat

,label,reward
0,0.0,-5.113281
1,0.0,-5.128906
2,0.0,-5.343750
3,0.0,-1.628906
4,0.0,-5.265625
...,...,...
21675,4.0,5.613281
21676,4.0,1.496094
21677,4.0,-0.365234
21678,4.0,2.191406


### Train Доля правильных reward для положительных отзывов(выбраны как таргетные)

In [21]:
stat[(stat["label"] > 1) & (stat["reward"]>0)].shape[0]/stat[(stat["label"] > 1)].shape[0]

0.967619926199262

### Доля правильных reward для отрицательных отзывов

In [22]:
stat[(stat["label"] <= 1) & (stat["reward"]<=0)].shape[0]/stat[(stat["label"] <= 1)].shape[0]

0.9618081180811808

In [32]:
ds = load_dataset("Yelp/yelp_review_full", split='test').filter(lambda example: example["label"]!=2 )

ds = ds.train_test_split(test_size=0.25, stratify_by_column="label")

dataset_test = ds["test"]

In [34]:
import collections
collections.Counter(dataset_test["label"])

Counter({1: 2500, 4: 2500, 0: 2500, 3: 2500})

In [35]:

label, rewards = reward_sanity_check(dataset_test)

stat = pd.DataFrame([label, rewards]).T.rename(columns={0:"label", 1:"reward"})
stat

,label,reward
0,1.0,-4.101562
1,4.0,5.726562
2,0.0,5.410156
3,3.0,4.570312
4,0.0,-4.769531
...,...,...
9995,3.0,5.714844
9996,4.0,5.718750
9997,1.0,-5.269531
9998,3.0,5.386719


### Test Доля правильных reward для положительных отзывов(выбраны как таргетные)

In [36]:
stat[(stat["label"] > 1) & (stat["reward"]>0)].shape[0]/stat[(stat["label"] > 1)].shape[0]

0.9488

### Доля правильных reward для отрицательных отзывов

In [37]:
stat[(stat["label"] <= 1) & (stat["reward"]<=0)].shape[0]/stat[(stat["label"] <= 1)].shape[0]

0.9414

### reward модель строилась как в примере, выше показал, что она хорошо назначает награды и на train и на test наборе.

# RL fine-tuning

### В качестве модели, которую будем файнтюнить выбрал "defex/distilgpt2-finetuned-amazon-reviews", так как она тоже предобучена на отзывах (но уже амазона). И при этом у нее нет смещения в сторону положительных или отрицательных отзывов.

In [38]:
from typing import List
def compute_reward(texts: List[str]) -> torch.Tensor:
  inputs = reward_tokenizer(texts, truncation=True, padding=True, return_tensors='pt').to(device)
  with torch.no_grad():
    return reward_model(**inputs).logits[:, 0]

In [54]:
import peft
peft_config = peft.LoraConfig(
    task_type=peft.TaskType.CAUSAL_LM, r=32, lora_alpha=32, lora_dropout=0.1, inference_mode=False
)

# reload main model as AutoModelForCausalLMWithValueHead - with an extra head needed for PPO
main_tokenizer = transformers.AutoTokenizer.from_pretrained("defex/distilgpt2-finetuned-amazon-reviews")
main_tokenizer.pad_token = main_tokenizer.eos_token

main_model = trl.AutoModelForCausalLMWithValueHead.from_pretrained("defex/distilgpt2-finetuned-amazon-reviews", device_map=device)
main_model = peft.get_peft_model(main_model, peft_config, adapter_name='default')
main_model.print_trainable_parameters()

trainable params: 589,824 || all params: 82,503,169 || trainable%: 0.714910720580927


In [41]:
# Note: this code is specific to IMDB; you will need to re-write it for other tasks

import torch
import transformers


import trl
review_for_rlhf = dataset.filter(lambda row: len(row['text']) < 200, batched=False)
review_for_rlhf = dataset.remove_columns(['label'])
sample_length = trl.core.LengthSampler(2, 8)  # use the first 2-8 tokens as query

def select_query_and_tokenize(sample):
    query_ids = main_tokenizer.encode(sample["text"])[: sample_length()]
    sample["query"] = main_tokenizer.decode(query_ids)  # query is the only required column
    sample["input_ids"] = query_ids  # to avoid re-tokenizing later
    return sample  # we do not need the rest - it will be generated by the model

review_for_rlhf = review_for_rlhf.map(select_query_and_tokenize, batched=False)
review_for_rlhf.set_format(type="torch")

Map:   0%|          | 0/21680 [00:00<?, ? examples/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (1232 > 1024). Running this sequence through the model will result in indexing errors


In [62]:
with open("./reward_review_model_checkpoint.pth", "rb") as fp:
    state_dict = torch.load(fp, map_location="cpu")
    reward_model.load_state_dict(state_dict)
    reward_model = reward_model.to(device)

In [55]:
training_args = trl.PPOConfig(
    model_name=main_model.config._name_or_path,
    gradient_accumulation_steps=1,
    learning_rate=1.41e-4,
    ppo_epochs=4,
    mini_batch_size=64,
    batch_size=64,# PPO performs this many updates per training batch
)

ppo_trainer = trl.PPOTrainer(
    training_args, model=main_model.model, tokenizer=main_tokenizer, ref_model=None,
    dataset=review_for_rlhf, data_collator=lambda data: dict((key, [d[key] for d in data]) for key in data[0])
)  # note: we pass main_model.model because PPOTrainer checks for one of several supported model types ...
# ... main_model.model is a model with adapters, which is supported. main_model itself is a wrapper that is not supported

In [56]:
from tqdm.auto import tqdm
max_steps = 20   # can be insufficient for some tasks - watch your learning curves
generation_kwargs = dict(
    min_length=-1, max_new_tokens=50, do_sample=True, top_k=0, top_p=1.0, pad_token_id=main_tokenizer.eos_token_id,
eos_token_id=main_tokenizer.eos_token_id,)
#                                  ^-- task-specific parameter!
with tqdm(enumerate(ppo_trainer.dataloader), total=max_steps) as progressbar:
  # note: ppo_trainer.dataloader is just a regular dataloader of queries, no RL-specific magic :)
  for epoch, batch in progressbar:
    if epoch >= max_steps:
        break

    # Rollout stage: generate continuations from batch queries using main_model
    response_tensors = ppo_trainer.generate(batch['input_ids'], **generation_kwargs)
    # ^-- list of tensors of token ids from main model tokenizer

    # de-tokenize responses to strings (since reward model uses a different tokenizer)
    batch["response"] = [main_tokenizer.decode(response.squeeze()) for response in response_tensors]
    # note: response_tensors already contain query tokens, so we don't need to add queries manually.
    # This may not be true for other tasks: check this manually by viewing batch["response"] and batch["query"]
    # Evaluation stage
    rewards = compute_reward(batch['response'])
    # Update stage
    stats = ppo_trainer.step(batch['input_ids'], response_tensors, list(rewards.split(1)))
    stats['rewards/mean'] = rewards.mean().item()

    print("-" * 30, 'STEP', epoch, '-' * 30)
    print(f'rewards/mean:\t{stats["rewards/mean"]:.9f}\t<---- average reward over this batch (higher=better, noisy)')
    print(f'ppo/returns/mean:\t{stats["ppo/returns/mean"]:.9f}\t<---- model-estimated average discounted reward')
    print(f'objective/kl:\t{stats["objective/kl"]:.9f}\t<---- how far we are from the original model (regularizer)')
    print()

    ppo_trainer.log_stats(stats, batch, list(rewards.split(1)))

  0%|          | 0/20 [00:00<?, ?it/s]

You're using a GPT2TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


------------------------------ STEP 0 ------------------------------
rewards/mean:	-0.281177521	<---- average reward over this batch (higher=better, noisy)
ppo/returns/mean:	0.553731918	<---- model-estimated average discounted reward
objective/kl:	0.000000000	<---- how far we are from the original model (regularizer)

------------------------------ STEP 1 ------------------------------
rewards/mean:	0.045897245	<---- average reward over this batch (higher=better, noisy)
ppo/returns/mean:	0.631238461	<---- model-estimated average discounted reward
objective/kl:	0.134747237	<---- how far we are from the original model (regularizer)

------------------------------ STEP 2 ------------------------------
rewards/mean:	-1.225526810	<---- average reward over this batch (higher=better, noisy)
ppo/returns/mean:	0.217734754	<---- model-estimated average discounted reward
objective/kl:	0.330352843	<---- how far we are from the original model (regularizer)

------------------------------ STEP 3 ---

In [57]:
ref_model = trl.AutoModelForCausalLMWithValueHead.from_pretrained("defex/distilgpt2-finetuned-amazon-reviews", device_map=device)
new = 0
old = 0
for i in range(10):

    inputs = main_tokenizer(["I think this product"] * 16, return_tensors='pt').to(device)

    new += compute_reward([main_tokenizer.decode(candidate.flatten().cpu().numpy().tolist()) for candidate in main_model.model.generate(**inputs, max_new_tokens=50, do_sample=True)]).mean()
    old += compute_reward([main_tokenizer.decode(candidate.flatten().cpu().numpy().tolist()) for candidate in ref_model.generate(**inputs, max_new_tokens=50, do_sample=True)]).mean()

Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Settin

### В качестве референсной модели берем "defex/distilgpt2-finetuned-amazon-reviews" и сравниваем средний reward до тюнинга с Lora и после. Разница есть:

In [58]:
print("mean reward before:", old/10)
print("mean reward after:", new/10)

mean reward before: tensor(-0.8653, device='cuda:0')
mean reward after: tensor(4.0462, device='cuda:0')


In [62]:
new = 0
old = 0
for i in range(10):

    inputs = main_tokenizer(["In my view"] * 16, return_tensors='pt').to(device)

    new += compute_reward([main_tokenizer.decode(candidate.flatten().cpu().numpy().tolist()) for candidate in main_model.model.generate(**inputs, max_new_tokens=50, do_sample=True)]).mean()
    old += compute_reward([main_tokenizer.decode(candidate.flatten().cpu().numpy().tolist()) for candidate in ref_model.generate(**inputs, max_new_tokens=50, do_sample=True)]).mean()

Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Settin

In [63]:
print("mean reward before:", old/10)
print("mean reward after:", new/10)

mean reward before: tensor(-0.7979, device='cuda:0')
mean reward after: tensor(3.9456, device='cuda:0')


In [65]:
new = 0
old = 0
for i in range(10):

    inputs = main_tokenizer(["I would say"] * 16, return_tensors='pt').to(device)

    new += compute_reward([main_tokenizer.decode(candidate.flatten().cpu().numpy().tolist()) for candidate in main_model.model.generate(**inputs, max_new_tokens=50, do_sample=True)]).mean()
    old += compute_reward([main_tokenizer.decode(candidate.flatten().cpu().numpy().tolist()) for candidate in ref_model.generate(**inputs, max_new_tokens=50, do_sample=True)]).mean()

Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Settin

In [66]:
print("mean reward before:", old/10)
print("mean reward after:", new/10)

mean reward before: tensor(-0.3320, device='cuda:0')
mean reward after: tensor(4.3473, device='cuda:0')


In [72]:
new = 0
old = 0
for i in range(10):

    inputs = main_tokenizer(["There is"] * 16, return_tensors='pt').to(device)

    new += compute_reward([main_tokenizer.decode(candidate.flatten().cpu().numpy().tolist()) for candidate in main_model.model.generate(**inputs, max_new_tokens=50, do_sample=True)]).mean()
    old += compute_reward([main_tokenizer.decode(candidate.flatten().cpu().numpy().tolist()) for candidate in ref_model.generate(**inputs, max_new_tokens=50, do_sample=True)]).mean()

Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Settin

In [73]:
print("mean reward before:", old/10)
print("mean reward after:", new/10)

mean reward before: tensor(-0.2131, device='cuda:0')
mean reward after: tensor(3.4256, device='cuda:0')


In [76]:
new = 0
old = 0
for i in range(10):

    inputs = main_tokenizer(["I can"] * 16, return_tensors='pt').to(device)

    new += compute_reward([main_tokenizer.decode(candidate.flatten().cpu().numpy().tolist()) for candidate in main_model.model.generate(**inputs, max_new_tokens=50, do_sample=True)]).mean()
    old += compute_reward([main_tokenizer.decode(candidate.flatten().cpu().numpy().tolist()) for candidate in ref_model.generate(**inputs, max_new_tokens=50, do_sample=True)]).mean()

Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Settin

In [77]:
print("mean reward before:", old/10)
print("mean reward after:", new/10)

mean reward before: tensor(-0.2783, device='cuda:0')
mean reward after: tensor(4.0723, device='cuda:0')


### Отзывы, которая генерит модель после finetune более позитивные, как и ожидалось:

In [59]:
inputs = main_tokenizer(["I think this product"] * 16, return_tensors='pt').to(device)

[main_tokenizer.decode(candidate.flatten().cpu().numpy().tolist()) for candidate in main_model.model.generate(**inputs, max_new_tokens=50, do_sample=True)]

Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


['I think this product is really helping with a high level of health, if you have any issues with the stomach it will be really helpful.It’s made for what you need in your car. It does take awhile to build up.Cute outfit. Super',
 'I think this product is really well made as well. And I would definitely buy it again as I love this product.I love it, I have to buy another book. I will look into other books and learn to like it, love it and give this amazing gift',
 'I think this product is great. My baby is much stronger which works ok. The battery does not go all the way to the side, so it is no easy to put together. So far, I like it so far.Easy to install with use. Perfect for',
 'I think this product is so sturdy. My kid loved it so much!I was very happy with this purchase even as it is a great product! And the quality of the knife is good!!I ordered these for a gift. The first one came broken when he took',
 'I think this product is a good replacement for my face itchy, but you can

In [67]:
inputs = main_tokenizer(["In my view"] * 16, return_tensors='pt').to(device)

[main_tokenizer.decode(candidate.flatten().cpu().numpy().tolist()) for candidate in main_model.model.generate(**inputs, max_new_tokens=50, do_sample=True)]

Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


['In my view the whole concept goes at a great value in design and product. The main thing is the lights are very clear that I bought them because they are very bright. I also used them for camping from the beach to a lot. I also like their flexibility',
 "In my view all this is a pretty solid product.The little white tape that sticks into your ears has been helpful from my husband's journey. It is so soft!This is a perfect gift! We have several pairs of earbuds that stay in and stay",
 "In my view it is much smaller than the standard. My kids use it daily for 3 hours everyday but these are very thin and fit nicely. I'd recommend these for all schoolwork. I would recommend that they fit and not keep your ears warm and ears warm",
 "In my view... I have to use it to make a big bowl to toast it in this style. This bag is made of very good quality ingredients. It is so versatile and a great weight.So far I'm not convinced that these are anything different than a",
 'In my view of these, y

In [68]:
inputs = main_tokenizer(["I would say"] * 16, return_tensors='pt').to(device)

[main_tokenizer.decode(candidate.flatten().cpu().numpy().tolist()) for candidate in main_model.model.generate(**inputs, max_new_tokens=50, do_sample=True)]

Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


['I would say the quality and the price was good for all of us, just wish it came preinstalled.Easy application to a new job.I really enjoy it. No problem.It was an easy read book, I enjoyed reading it.Very durable. I',
 "I would say it is the best hairline mascara I use. It does the job of removing my eyeliner but it is a little thick in hair.Not overly impressed with the quality of this product.Works great! It's perfect with my phone but i really",
 'I would say this book gave me one of the best and super entertaining series of my adult life!This is exactly what it is. Very easy to read and a great way to set up I love so very much! It has great taste and is a wonderful addition',
 'I would say this is a good product as well as the tools. It is a little heavy, but the light weight is great for my work and my wife is really handy.Works great.Great price so far so happy with the product.I love the recipes',
 'I would say the product works. The way it is used to keep the water cold and 

In [74]:
inputs = main_tokenizer(["There is"] * 16, return_tensors='pt').to(device)

[main_tokenizer.decode(candidate.flatten().cpu().numpy().tolist()) for candidate in main_model.model.generate(**inputs, max_new_tokens=50, do_sample=True)]

Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


["There is a small space. Easy to assemble.Great for what you need it for to work and it is as pictured. Only problem is its very light to it.Doesn't like the price!This is a great product. I can’t",
 'There is a good way to use your hands.I used this for a vacation or picnic and this product was my go hand and it arrived promptly.These bags are nice but the price is too expensive. They are a little thicker than some other bags I have',
 "There is a good case. But is not very heavy or strong enough to fit all my phones in the car. It's the best. I have had trouble sleeping this case and that has helped me. It really did get my mind spinning. We were not",
 'There is no odor or a soft finish.These headphones do what we need on a normal car ride. I just wish it was on longer.Great gift for my granddaughter. The only thing I do that she loves it so much.Came with instructions.',
 'There is always a problem with the first ones I used when a little bit stiff. But not in the back yet, this i

In [78]:
inputs = main_tokenizer(["I can"] * 16, return_tensors='pt').to(device)

[main_tokenizer.decode(candidate.flatten().cpu().numpy().tolist()) for candidate in main_model.model.generate(**inputs, max_new_tokens=50, do_sample=True)]

Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


["I can't understand the other flavors (like chocolate), so I think that I won't be able to understand them for a while, so I will try that at later. The flavor is good.I love all 3 flavors, it is very sweet and perfect",
 'I can think of many ways to control your posture and how it can assist the child. The concept of this set was so nice and simple to get off it!I like you to use it. You get what you pay for.I just installed in my',
 "I can see it is being used by some. I can see it going with a little weight but the fact that it's so light and does look neat, is worth the price.Good battery, works as advertised. I used it as both a battery and",
 "I can see the difference in other ways that I use the product. Very happy with this product. I have one more bag for use in the office to have more space for cooking that I have that's more convenient for me and a little larger than I usually",
 'I can see where the sun goes and if I can do anything with this, I will go ahead and purchase 

### Отзывы, которая генерит модель до finetune более разношерстные(выше доля негативных):

In [61]:
inputs = main_tokenizer(["I think this product"] * 16, return_tensors='pt').to(device)

[main_tokenizer.decode(candidate.flatten().cpu().numpy().tolist()) for candidate in ref_model.generate(**inputs, max_new_tokens=50, do_sample=True)]

Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


["I think this product will help me sleep. Not sure if I have any issues going forward.My wife doesn't like it, the packaging goes weird every couple of days. The color in this pack looks a little strange. I had more of a blue color that was",
 "I think this product is not as useful as a shampoo line, so maybe I am wrong. Very nice and inexpensive.I can't use it without the other hand, and these are the best. They're smaller than expected, and are not a fan.Works as",
 'I think this product is well made and was only used once but it will last a week before breaking off and cracking. Also, the box came with a 2 inch jargie, even though it looked like it would be ok. I had other jars I was hoping for',
 "I think this product really is helping me to get people's attention. I do believe this is a really good product to really help them, because it's been working great for me. I have used so many times, I can feel the difference in my skin, I",
 'I think this product might be working well f

In [69]:
inputs = main_tokenizer(["I would say"] * 16, return_tensors='pt').to(device)

[main_tokenizer.decode(candidate.flatten().cpu().numpy().tolist()) for candidate in main_model.model.generate(**inputs, max_new_tokens=50, do_sample=True)]

Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


["I would say they're the best.Love these glasses! Great for my husband looking to make homemade beverages. Not too bulky and comfortable just for one bottle. Not too bulky just to fit right.Works well for working with my kidsI am impressed by this product",
 'I would say this movie was a good way to make a movie about something but so did the premise and that is what i loved about it.It was a good workout routine and very well-made. My son is 6 months old when we get a run at',
 'I would say this review was a very well written, well researched, informative, thorough read book and is a good read to read.This is a nice product and fits perfectly. My dog absolutely loves the extra soft drinks and this helps her a lot as she loves',
 "I would say that these did NOT fit my hair on the phone. They are a little shorter than I expected.Not very sturdy and makes the job a little easier. But this is my 5th Birthday and it's been a great season since. I haven't",
 'I would say that this is a qua

In [71]:
inputs = main_tokenizer(["I would say"] * 16, return_tensors='pt').to(device)

[main_tokenizer.decode(candidate.flatten().cpu().numpy().tolist()) for candidate in ref_model.generate(**inputs, max_new_tokens=50, do_sample=True)]

Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


['I would say the color of this bottle is very nice. It seems nice. The black ink color does not like the quality and consistency. However, it’s not quite the one I bought from a different retailer. It’s a medium size bottle',
 'I would say the screen protector is a little more bulky than I would like. It fits the phone with a case and there is enough room in the case on my watch to take and look at it for a few hours. I don’t feel like wasting',
 'I would say I’ve enjoyed the series pretty much without the actors. A wonderful book.I think i got one to rate the reviews on this brand because I like the idea of these things, but one of the other items that I was happy with was',
 'I would say the only downside was that it did not stick well. It would not stick to the car. The screen protector was already broken after about 8 months of use.Didn’t get what I paidI ordered the same size up and ordered this',
 "I would say that it would be great if it could hold more paper, but, as time goes

In [75]:
inputs = main_tokenizer(["There is"] * 16, return_tensors='pt').to(device)

[main_tokenizer.decode(candidate.flatten().cpu().numpy().tolist()) for candidate in ref_model.generate(**inputs, max_new_tokens=50, do_sample=True)]

Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


['There is a bit hard to find a good and durable product.I use these as a tool to install oil filters. The most beautiful thing about these is they do not stop the flow and get hot in the shower. I had no issues with heat.This',
 'There is this table top with a picture of the contents of the top making it very inconvenient to look at the full contents. I ended up ordering 2 items not very useful. All the items I needed were used, and the picture showed the table front. As',
 'There is no way to see if it is a big deal for a smaller brand but I could see it being a larger sizeThis was a great case for my iPad Pro, I have small hands, but my MacBook Pro 8 Pro 9 is an XL. It',
 'There is a good set of lights that can be adjusted and controlled and work great for any setting. This is super easy to install and clean. Love it!Works great in my office but one problem is the wire came apart. Once it’s together',
 "There is no help and is basically worthlessI have had this since Christmas and i

In [79]:
inputs = main_tokenizer(["I can"] * 16, return_tensors='pt').to(device)

[main_tokenizer.decode(candidate.flatten().cpu().numpy().tolist()) for candidate in ref_model.generate(**inputs, max_new_tokens=50, do_sample=True)]

Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


["I can't think of a better way to get the item shipped. I guess I'm going to keep it because I am so very disappointed in it!The price for this product is great. It comes with stickers on the side, but that’s",
 "I can't see it coming in a good way.I bought these for my daughter and her mother after Christmas! She loves them!!! I am so grateful I got my daughter to wear those! They are a bit stretchy and the fit is very nice!",
 "I can't wait to get the bottle in. And this is the last bottle we will have to put this bottles out. So much fun when these are in a box full of stuff.Awesome but a little stiff. Looks to be a sturdy design.Nice",
 "I can't remember much about this book... I read the first three parts, but it's still a great novel!I thought it was the product I was looking for.Very smallI love the size, although the fabric is a little tight, so I",
 'I can’t stand this. It is too difficult to move and it is difficult to keep on the chair. If you just want to keep on your chai